# Moteur de Recherche Sémantique avec Catégorisation du Type de Question

## Objectif

Ce notebook implémente un assistant QA pédagogique qui, à partir d'une question posée par un étudiant :

1. **Vérifie que la question est liée au cours** via un seuil sémantique
2. **Catégorise le type de la question** (définition, formule, exemple, remarque, image)
3. **Filtre les slides** qui correspondent à ce type
4. **Extrait la meilleure réponse** dans ces slides
5. **Répond "je ne connais pas la réponse"** si aucun slide pertinent n'est trouvé

## Pourquoi catégoriser le type de question ?

Chaque slide du cours est étiqueté avec une catégorie :

| Catégorie    | Exemples de questions                                      |
|--------------|------------------------------------------------------------|
| `definition` | "Qu'est-ce que… ?", "Définir…", "C'est quoi… ?"           |
| `formule`    | "Quelle est la formule de… ?", "Comment calcule-t-on… ?"  |
| `exemple`    | "Donne un exemple de…", "Illustrer… par un exemple"        |
| `remarque`   | "Quelles sont les propriétés de… ?", "Que remarque-t-on…" |
| `image`      | "Montre le schéma de…", "Représentation graphique de…"    |

En filtrant d'abord par type, on réduit le bruit et on améliore la précision de la réponse.
Si aucun slide de la catégorie détectée ne dépasse le seuil de confiance, la recherche
s'élargit automatiquement à tous les slides. Si la catégorie est inconnue, la recherche
porte d'emblée sur l'ensemble du corpus.

# Chargement des données

In [31]:
import json
import re
import os
os.environ["TQDM_DISABLE"] = "1"

# Force tqdm to use plain text mode - must happen before sentence_transformers import
from tqdm.auto import tqdm as tqdm_auto
from tqdm import tqdm as tqdm_std
import tqdm.auto
tqdm.auto.tqdm = tqdm_std  # replace notebook/widget tqdm with plain text tqdm

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("intfloat/multilingual-e5-base")
print("Modèle chargé.")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1742.81it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modèle chargé.


In [32]:
slide_embeddings = np.load("../data/3-numerical/slide_embeddings.npy")
print(f"Embeddings du cours chargés ({slide_embeddings.shape[0]} slides, dim {slide_embeddings.shape[1]}).")

with open("../data/3-numerical/slides_metadata.json", "r", encoding="utf-8") as f:
    slides_metadata = json.load(f)

print(f"Métadonnées chargées ({len(slides_metadata)} slides).")

Embeddings du cours chargés (75 slides, dim 768).
Métadonnées chargées (75 slides).


# Fonctions

In [33]:
import re
import unicodedata

def normalize(text: str) -> str:

    text = text.lower()

    text = ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [34]:
CATEGORIES = {

    "definition": [
        "definir",
        "definition",
        "qu'est ce que",
        "qu est ce que",
        "qu'est-ce qu'",
        "c'est quoi",
        "c est quoi",
        "que signifie",
        "signifie"
    ],

    "formule": [
        "formule",
        "equation",
        "calcul",
        "calculer",
        "expression",
        "relation"
    ],

    "exemple": [
        "exemple",
        "illustrer",
        "application",
        "cas pratique"
    ],

    "remarque": [
        "remarque",
        "attention",
        "important",
        "particularite",
        "limitation"
    ],

    "image": [
        "image",
        "figure",
        "schema",
        "graphe",
        "graphique",
        "courbe"
    ]
}


def classify_question(question: str) -> str:

    q = normalize(question)

    for category, keywords in CATEGORIES.items():

        for kw in keywords:

            if normalize(kw) in q:
                return category

    return "unknown"

In [35]:
def split_sentences(text: str) -> list:

    if not text:
        return []

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text.strip()
    )

    sentences = [
        re.sub(r'\s+', ' ', s).strip()
        for s in sentences
        if s.strip()
    ]

    return sentences

In [36]:
SALUTATIONS = {
    "bonjour", "bonsoir", "salut", "hello", "hi", "hey",
    "salam", "coucou", "yo", "allo", "allô"
}

def is_salutation(question: str) -> bool:
    """Détecte si la question est une simple salutation."""
    q = normalize(question)
    tokens = q.split()
    return any(token in SALUTATIONS for token in tokens)

In [40]:
import numpy as np

def repondre(
    question: str,
    top_k: int = 1,
    use_sentence_extraction: bool = True,
    slide_threshold: float = 0.75,
    phrase_threshold: float = 0.65
) -> dict:

    # -----------------------------------------------------------------------
    # Détection des salutations
    # -----------------------------------------------------------------------

    if is_salutation(question):
        return {
            "chap_id": None, "chap_name": None, "slide_id": None,
            "section": None, "category": "salutation", "score": 0.0,
            "reponse": "Hello ! Je suis votre assistant pédagogique. Posez-moi une question sur le cours."
        }

    # --------------------------------------------------
    # Encodage question (une seule fois)
    # --------------------------------------------------

    q_emb = model.encode(
        [f"query: {question}"],
        normalize_embeddings=True
    )

    # --------------------------------------------------
    # Rejet hors domaine — sémantique, indépendant de la catégorie
    # --------------------------------------------------

    max_score = float(np.dot(slide_embeddings, q_emb.T).max())
    if max_score < 0.8:
        return {
            "chap_id": None,
            "chap_name": None,
            "slide_id": None,
            "section": None,
            "category": "unknown",
            "score": 0.0,
            "reponse": "Je ne connais pas la réponse dans le cours."
        }

    # --------------------------------------------------
    # Catégorisation — sert uniquement au filtrage, jamais au rejet
    # --------------------------------------------------

    question_category = classify_question(question)

    # --------------------------------------------------
    # Filtrage par catégorie
    # --------------------------------------------------

    if question_category != "unknown":
        candidate_indices = [
            i for i, slide in enumerate(slides_metadata)
            if slide.get("category") == question_category
        ]
        if len(candidate_indices) == 0:
            candidate_indices = list(range(len(slides_metadata)))
    else:
        candidate_indices = list(range(len(slides_metadata)))

    # --------------------------------------------------
    # Similarité des slides
    # --------------------------------------------------

    candidate_embeddings = slide_embeddings[candidate_indices]
    scores = np.dot(candidate_embeddings, q_emb.T).flatten()
    best_local_idx = scores.argmax()
    best_slide_score = float(scores[best_local_idx])

    # --------------------------------------------------
    # Fallback : si score insuffisant dans la catégorie,
    # on élargit à tous les slides avant de rejeter
    # --------------------------------------------------

    if best_slide_score < slide_threshold and len(candidate_indices) < len(slides_metadata):
        candidate_indices = list(range(len(slides_metadata)))
        candidate_embeddings = slide_embeddings[candidate_indices]
        scores = np.dot(candidate_embeddings, q_emb.T).flatten()
        best_local_idx = scores.argmax()
        best_slide_score = float(scores[best_local_idx])

    # --------------------------------------------------
    # Rejet final — seulement après avoir cherché partout
    # --------------------------------------------------

    if best_slide_score < slide_threshold:
        return {
            "chap_id": None,
            "chap_name": None,
            "slide_id": None,
            "section": None,
            "category": question_category,
            "score": best_slide_score,
            "reponse": "Je ne connais pas la réponse dans le cours."
        }

    # --------------------------------------------------
    # Top K
    # --------------------------------------------------

    best_local_indices = scores.argsort()[::-1][:top_k]
    resultats = []

    for local_idx in best_local_indices:

        idx = candidate_indices[local_idx]
        slide = slides_metadata[idx]
        content = slide.get("slide_content", "")
        slide_score = float(scores[local_idx])

        if not content:
            reponse = "[Pas de texte exploitable]"
            phrase_score = 0.0

        elif use_sentence_extraction:

            phrases = split_sentences(content)

            if len(phrases) == 0:
                reponse = content[:300]
                phrase_score = slide_score

            else:
                phr_embs = model.encode(
                    [f"passage: {p}" for p in phrases],
                    normalize_embeddings=True
                )
                phr_scores = np.dot(phr_embs, q_emb.T).flatten()
                best_phrase_idx = phr_scores.argmax()
                phrase_score = float(phr_scores[best_phrase_idx])
                reponse = phrases[best_phrase_idx]

        else:
            reponse = content[:500]
            if len(content) > 500:
                reponse += "..."
            phrase_score = slide_score

        if phrase_score < phrase_threshold:
            continue

        resultats.append({
            "chap_id": slide.get("chap_id"),
            "chap_name": slide.get("chap_name"),
            "slide_id": slide.get("slide_id"),
            "section": slide.get("section"),
            "category": question_category,
            "score": round(phrase_score, 4),
            "reponse": reponse
        })

    if not resultats:
        return {
            "chap_id": None,
            "chap_name": None,
            "slide_id": None,
            "section": None,
            "category": question_category,
            "score": 0.0,
            "reponse": "Je ne connais pas la réponse dans le cours."
        }

    return resultats[0]

# Tests

In [41]:
def tester(question):

    rep = repondre(question)

    print("\n" + "-" * 50)
    print(f"Question   : {question}")
    print(f"Catégorie  : {rep.get('category', 'unknown')}")
    print(f"Score      : {rep['score']:.4f}")

    if rep["chap_id"] is not None:
        print(f"Chapitre   : {rep['chap_id']} - {rep['chap_name']}")
        print(f"Slide      : {rep['slide_id']} - {rep['section']}")

    print(f"Réponse    : {rep['reponse']}")
if __name__ == "__main__":

    tester("Qu'est-ce qu'un signal périodique ?")

    tester("Définir produit de convolution")
    tester("Qui est Diego Armando Maradona ?")
    tester("Définition de l'énergie d'un signal")

    tester("C'est quoi mono-variable ?")

    # Question volontairement hors sujet
    tester("coucou")
    tester("Qui a gagné la Coupe du monde de football 2018 ?")
    tester("Quelle est la capitale du Japon ?")
    tester("Qui est Albert Einstein ?")


--------------------------------------------------
Question   : Qu'est-ce qu'un signal périodique ?
Catégorie  : definition
Score      : 0.8729
Chapitre   : 1 - Représentation temporelle des signaux
Slide      : 19 - Classification énergétique
Réponse    : Propriétés d’un signal périodique: Un signal périodique continu a nécessairement un support temporel non borné.

--------------------------------------------------
Question   : Définir produit de convolution
Catégorie  : definition
Score      : 0.8850
Chapitre   : 2 - Systèmes linéaires continus
Slide      : 22 - Produit de convolution
Réponse    : On appelle Produit de convolution de deux signaux [EQ: 𝑥(𝑡)] et [EQ: 𝑦(𝑡)], et on note [EQ: 𝑥(𝑡)∗𝑦(𝑡)] l’expression: Cas de signaux causaux: [EQ: 𝑥(𝑡)=0, 𝑦(𝑡)=0 pour 𝑡 <0]

--------------------------------------------------
Question   : Qui est Diego Armando Maradona ?
Catégorie  : unknown
Score      : 0.0000
Réponse    : Je ne connais pas la réponse dans le cours.

----------------------